In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [9]:
file_path = "dataset/komoditas_daging_sapi_2022_2026.csv"
df = pd.read_csv(file_path)

In [10]:
# Menampilkan 5 data awal untuk memastikan data terbaca
print("=== 5 DATA AWAL ===")
print(df.head())

# Memilih kolom yang penting untuk analisis
# Kita gunakan:
# - Date_Param sebagai tanggal
# - Province_Name sebagai provinsi
# - Price sebagai target harga
df = df[["Date_Param", "Province_Name", "Price"]]

# Mengubah kolom tanggal ke format datetime
df["Date_Param"] = pd.to_datetime(df["Date_Param"])

# Membuat fitur dari tanggal
# Fitur ini digunakan agar model dapat membaca pola waktu
df["Tahun"] = df["Date_Param"].dt.year
df["Bulan"] = df["Date_Param"].dt.month
df["Hari"] = df["Date_Param"].dt.day
df["Hari_dalam_Minggu"] = df["Date_Param"].dt.dayofweek

# Mengubah nama provinsi menjadi angka
# Model machine learning hanya bisa membaca data numerik
df["Kode_Provinsi"] = df["Province_Name"].astype("category").cat.codes

# Menampilkan informasi dataset
print("\n=== INFO DATASET ===")
print(df.info())

print("\n=== JUMLAH DATA ===")
print(df.shape)

=== 5 DATA AWAL ===
  Date_Scraped  Date_Param  Commodity_ID Commodity_Name  Province_ID  \
0   2026-02-12  2022-01-01             3    Daging Sapi            1   
1   2026-02-12  2022-01-01             3    Daging Sapi            2   
2   2026-02-12  2022-01-01             3    Daging Sapi            3   
3   2026-02-12  2022-01-01             3    Daging Sapi            4   
4   2026-02-12  2022-01-01             3    Daging Sapi            5   

    Province_Name     Price  Price_Type  
0            Aceh  140400.0           1  
1  Sumatera Utara  128150.0           1  
2  Sumatera Barat  135000.0           1  
3            Riau  117900.0           1  
4  Kepulauan Riau  112450.0           1  

=== INFO DATASET ===
<class 'pandas.DataFrame'>
RangeIndex: 47210 entries, 0 to 47209
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date_Param         47210 non-null  datetime64[us]
 1   Provin

In [ ]:
# Menentukan fitur (X) dan target (y)
# X = data input
# y = harga yang akan diprediksi
X = df[["Kode_Provinsi", "Tahun", "Bulan", "Hari", "Hari_dalam_Minggu"]]
y = df["Price"]

# Membagi data menjadi data latih dan data uji
# 80% data latih dan 20% data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Membuat model Random Forest Regressor
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Melatih model menggunakan data latih
model.fit(X_train, y_train)

# Melakukan prediksi pada data uji
y_pred = model.predict(X_test)

# Menghitung evaluasi model
# MAE  = Mean Absolute Error
# RMSE = Root Mean Squared Error
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n=== EVALUASI MODEL ===")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))

# Menampilkan contoh hasil prediksi
hasil_uji = pd.DataFrame({
    "Harga Asli": y_test.values[:10],
    "Harga Prediksi": y_pred[:10]
})

print("\n=== CONTOH HASIL PREDIKSI ===")
print(hasil_uji)



=== EVALUASI MODEL ===
MAE  : 277.98
RMSE : 1066.15

=== CONTOH HASIL PREDIKSI ===
   Harga Asli  Harga Prediksi
0    119800.0        119793.5
1    132500.0        133187.5
2    147300.0        147348.5
3    151400.0        139789.5
4    136250.0        136225.5
5    131900.0        132554.5
6    153450.0        153675.0
7    143350.0        143375.0
8    131250.0        130344.0
9    126800.0        127908.0


In [13]:
# Membuat mapping kode provinsi
# Agar user tahu angka mana untuk provinsi tertentu
mapping_provinsi = dict(enumerate(df["Province_Name"].astype("category").cat.categories))

print("\n=== DAFTAR KODE PROVINSI ===")
for kode, prov in mapping_provinsi.items():
    print(f"{kode} : {prov}")# Membuat mapping kode provinsi

# Input data baru dari pengguna
# User memilih provinsi dan tanggal
print("\n=== PREDIKSI HARGA BARU ===")
kode_provinsi = int(input("Masukkan kode provinsi: "))
tahun = int(input("Masukkan tahun: "))
bulan = int(input("Masukkan bulan: "))
hari = int(input("Masukkan hari: "))

# Menghitung hari dalam minggu dari tanggal input
tanggal_baru = pd.Timestamp(year=tahun, month=bulan, day=hari)
hari_dalam_minggu = tanggal_baru.dayofweek

# Menyiapkan data baru ke dalam DataFrame
data_baru = pd.DataFrame([{
    "Kode_Provinsi": kode_provinsi,
    "Tahun": tahun,
    "Bulan": bulan,
    "Hari": hari,
    "Hari_dalam_Minggu": hari_dalam_minggu
}])

# Melakukan prediksi harga untuk data baru
prediksi_baru = model.predict(data_baru)[0]

# Menampilkan hasil prediksi akhir
print("\n=== HASIL PREDIKSI BARU ===")
print("Provinsi        :", mapping_provinsi.get(kode_provinsi, "Tidak diketahui"))
print("Tanggal         :", tanggal_baru.date())
print("Prediksi Harga  :", round(prediksi_baru, 2))
print("Komoditas       : Daging Sapi")


=== DAFTAR KODE PROVINSI ===
0 : Aceh
1 : Bali
2 : Banten
3 : Bengkulu
4 : DI Yogyakarta
5 : DKI Jakarta
6 : Gorontalo
7 : Jambi
8 : Jawa Barat
9 : Jawa Tengah
10 : Jawa Timur
11 : Kalimantan Barat
12 : Kalimantan Selatan
13 : Kalimantan Tengah
14 : Kalimantan Timur
15 : Kalimantan Utara
16 : Kepulauan Bangka Belitung
17 : Kepulauan Riau
18 : Lampung
19 : Maluku
20 : Maluku Utara
21 : Nusa Tenggara Barat
22 : Nusa Tenggara Timur
23 : Papua
24 : Papua Barat
25 : Riau
26 : Sulawesi Barat
27 : Sulawesi Selatan
28 : Sulawesi Tengah
29 : Sulawesi Tenggara
30 : Sulawesi Utara
31 : Sumatera Barat
32 : Sumatera Selatan
33 : Sumatera Utara

=== PREDIKSI HARGA BARU ===

=== HASIL PREDIKSI BARU ===
Provinsi        : Sulawesi Tenggara
Tanggal         : 2030-11-25
Prediksi Harga  : 136900.0
Komoditas       : Daging Sapi
